# Alibaba model training on Colab T4
Set the runtime to T4 GPU. Upload a ZIP containing the `Alibaba_Model` folder when prompted. The notebook installs isolated dependencies, runs all stages, and downloads a ZIP with the generated artifacts, EDA, and processed splits.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys
uploaded = files.upload()
archive = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if archive is None: raise ValueError('Upload a ZIP containing Alibaba_Model/')
workspace = Path('/content/alibaba_workspace')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as source:
    if any('..' in Path(name).parts or Path(name).is_absolute() for name in source.namelist()): raise ValueError('Unsafe ZIP path')
    source.extractall(workspace)
root = next(path for path in workspace.rglob('config.yaml') if path.parent.name == 'Alibaba_Model').parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root/'requirements.txt')], check=True)
subprocess.run([sys.executable, str(root/'src/run_pipeline.py'), '--device', 'cuda'], cwd=root, check=True)
result = Path('/content/alibaba_training_results.zip')
with zipfile.ZipFile(result, 'w', zipfile.ZIP_DEFLATED) as target:
    for directory in ('artifacts', 'eda', 'data/processed'):
        for path in (root/directory).rglob('*'):
            if path.is_file(): target.write(path, path.relative_to(root))
files.download(str(result))
